In [ ]:
# ============================================================
# SPOTIFY CRIMSON ULTRA DASHBOARD - GOOGLE COLAB PRODUCTION
# POWER BI GRID DESIGN MAP IN THEME GREEN (REAL DATA PIPELINE)
# ============================================================

# =========================
# STEP 1 : INSTALL LIBRARIES
# =========================
!pip install dash plotly pandas -q

# =========================
# STEP 2 : IMPORT LIBRARIES
# =========================
import os
import pandas as pd
import numpy as np
from dash import Dash, dcc, html
import plotly.express as px
import plotly.graph_objects as go
from google.colab import output
import threading

# ============================================================
# --- STEP 3: AUTOMATED DATA PIPELINE (REAL SPOTIFY CSV) ---
# ============================================================
# Checks both local directory and Colab environment paths
SPOTIFY_CSV_PATH = 'spotify.csv' if os.path.exists('spotify.csv') else '/content/spotify.csv'

if os.path.exists(SPOTIFY_CSV_PATH):
    print(f"✅ Real data source discovered! Loading from: {SPOTIFY_CSV_PATH}")
    df = pd.read_csv(SPOTIFY_CSV_PATH)
else:
    print(f"⚠️ '{SPOTIFY_CSV_PATH}' not detected. Creating clean runtime baseline fallback...")
    dates_fallback = pd.date_range(start='2017-01-06', end='2017-12-31')
    df = pd.DataFrame({
        'Date': dates_fallback,
        'Shape of You': np.random.randint(2000000, 15000000, len(dates_fallback)),
        'Despacito': np.random.randint(1000000, 12000000, len(dates_fallback)),
        'Something Just Like This': np.random.randint(800000, 8000000, len(dates_fallback)),
        'HUMBLE.': np.random.randint(500000, 6000000, len(dates_fallback)),
        'Unforgettable': np.random.randint(400000, 5000000, len(dates_fallback))
    })

# Format explicit structural dates & handle missing values
df['Date'] = pd.to_datetime(df['Date'])
track_columns = [col for col in df.columns if col != 'Date']
df_filled = df.copy()
df_filled[track_columns] = df_filled[track_columns].fillna(0)

# Live Dynamic KPI Computations
total_streams_series = df_filled[track_columns].sum()
grand_total_calculated = total_streams_series.sum()
top_performing_track = total_streams_series.idxmax()
highest_volume_track_count = total_streams_series.max()

# Metrics mapped to the Power BI card text system matching layout blueprint
total_titles = f"{grand_total_calculated:,.0f}"
movies_count = f"{top_performing_track}"
tv_seasons = f"{highest_volume_track_count:,.0f}"
territories = f"{len(track_columns)}"

# =========================
# STEP 4 : COLOR SYSTEM CONSTANTS (SPOTIFY PREMIUM THEME)
# =========================
CANVAS_BG = '#0A0A0A'       # Premium pitch black canvas
CARD_BG = '#121212'         # Spotify dark grey card body
BORDER_COLOR = '#282828'    # Sleek dark panel dividers
SPOTIFY_GREEN = '#1DB954'   # Core iconic Spotify Green
TEXT_MAIN = '#FFFFFF'       # High contrast white text
TEXT_MUTED = '#B3B3B3'      # Spotify premium grey secondary text

# Premium Spotify green tonal gradient scale for columns and charts
GREEN_GRADIENT = ['#1DB954', '#1ED760', '#1aa34a', '#148a3e', '#0f6e31', '#0b5425']

# Panel structure stylesheet mapping
panel_layout_style = {
    'backgroundColor': CARD_BG,
    'padding': '16px',
    'borderRadius': '8px',
    'border': f'1px solid {BORDER_COLOR}',
    'boxShadow': '0px 4px 25px rgba(0, 0, 0, 0.5)',
    'marginBottom': '15px'
}

kpi_panel_style = {
    **panel_layout_style,
    'textAlign': 'center',
    'flex': '1',
    'minWidth': '120px',
    'marginBottom': '0px',
    'padding': '12px 6px',
    'borderTop': f'3px solid {SPOTIFY_GREEN}'  # Power BI Styled Spotify Green top accent bar
}

# Reusable chart style framework engine
def format_spotify_chart(fig, title_text):
    fig.update_layout(
        title=dict(text=title_text, font=dict(size=14, color=TEXT_MAIN, family='Arial')),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font_color=TEXT_MAIN,
        margin=dict(l=65, r=20, t=45, b=35),
        showlegend=False
    )
    fig.update_xaxes(showgrid=False, color=TEXT_MUTED, title='')
    fig.update_yaxes(showgrid=False, color=TEXT_MUTED, title='')
    return fig

# =========================
# STEP 5 : CHARTS IMPLEMENTATION
# =========================

# 1. Track Playback Share Allocation (Horizontal Bars)
ranking_df = total_streams_series.reset_index()
ranking_df.columns = ['Track', 'Streams']
ranking_df = ranking_df.sort_values('Streams', ascending=True)
fig_state = px.bar(ranking_df, x='Streams', y='Track', orientation='h', color_discrete_sequence=[SPOTIFY_GREEN])
format_spotify_chart(fig_state, "Sum of Playbacks by Track Name")

# 2. Cumulative Composition (Donut Chart)
fig_category = px.pie(ranking_df, values='Streams', names='Track', hole=0.6, color_discrete_sequence=GREEN_GRADIENT)
format_spotify_chart(fig_category, "Distribution Share by Catalog Release")
fig_category.update_traces(textposition='outside', textinfo='label+percent')

# 3. Monthly Stream Breakdown (Timeline Trend Paths)
monthly_df = df_filled.set_index('Date').resample('ME')[track_columns].sum().reset_index()
monthly_melted = monthly_df.melt(id_vars='Date', var_name='Track', value_name='Streams')
fig_subcat = px.line(monthly_melted, x='Date', y='Streams', color='Track', color_discrete_sequence=GREEN_GRADIENT)
format_spotify_chart(fig_subcat, "Streaming Trend Timeline (Monthly Aggregations)")
fig_subcat.update_traces(line=dict(width=2.5))

# 4. Top Performing Months - Column Standings (Multi-Colored Vertical Columns)
df_filled['MonthName'] = df_filled['Date'].dt.strftime('%b')
month_volume = df_filled.groupby('MonthName')[track_columns].sum().sum(axis=1).reset_index()
month_volume.columns = ['Month', 'Total Streams']
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
month_volume['Month'] = pd.Categorical(month_volume['Month'], categories=month_order, ordered=True)
month_volume = month_volume.sort_values('Month').head(6)

fig_customer = px.bar(month_volume, x='Month', y='Total Streams', color='Month', color_discrete_sequence=GREEN_GRADIENT)
format_spotify_chart(fig_customer, "Sum of Streams by Peak Calendar Months")

# 5. Track Peak Playback Benchmark Density (Pie / Donut Representation)
fig_payment = px.pie(ranking_df, values='Streams', names='Track', hole=0.6, color_discrete_sequence=GREEN_GRADIENT[::-1])
format_spotify_chart(fig_payment, "Benchmark Performance Concentration Density")
fig_payment.update_traces(textposition='inside', textinfo='percent+label', textfont_size=10)

# =========================
# STEP 6 : LAYOUT CONTAINER
# =========================
app = Dash(__name__)

slicer_btn = {
    'backgroundColor': '#282828', 'color': '#ffffff', 'border': '1px solid #404040',
    'padding': '4px 12px', 'fontSize': '11px', 'borderRadius': '4px', 'cursor': 'pointer'
}
slicer_active = {**slicer_btn, 'backgroundColor': SPOTIFY_GREEN, 'border': f'1px solid {SPOTIFY_GREEN}'}

app.layout = html.Div([

    # --- BRANDED OFFICIAL SPOTIFY TITLE HEADER ---
    html.Div([
        html.Div([
            html.Img(src="https://upload.wikimedia.org/wikipedia/commons/1/19/Spotify_logo_without_text.svg", style={'height': '35px', 'marginRight': '15px'}),
            html.Div([
                html.H1("SPOTIFY MUSIC ANALYTICS DASHBOARD", style={'color': TEXT_MAIN, 'margin': '0', 'fontSize': '22px', 'fontWeight': '900', 'fontFamily': 'Arial'}),
                html.P("Global Streaming Catalog Performance Analytics Pipeline", style={'color': TEXT_MUTED, 'margin': '0', 'fontSize': '12px'})
            ])
        ], style={'display': 'flex', 'alignItems': 'center'}),

        html.Div([
            html.Span("● ENGINE ONLINE", style={'color': '#1DB954', 'fontSize': '11px', 'fontWeight': 'bold', 'backgroundColor': 'rgba(29,185,84,0.1)', 'padding': '5px 12px', 'borderRadius': '20px', 'marginRight': '15px'}),
            html.Span("V4.2 Production Pipeline", style={'color': TEXT_MUTED, 'fontSize': '11px'})
        ], style={'display': 'flex', 'alignItems': 'center'})
    ], style={'display': 'flex', 'justifyContent': 'space-between', 'alignItems': 'center', 'borderBottom': '1px solid #282828', 'paddingBottom': '15px', 'marginBottom': '20px'}),

    # --- TOP CONTROL SUB CONTAINER: METRICS + SLICERS ---
    html.Div([
        # Dynamic Calculated KPI Blocks
        html.Div([
            html.Div([html.H2(total_titles, style={'color': TEXT_MAIN, 'margin': '0', 'fontSize': '18px'}), html.P("TOTAL COMBINED STREAMS", style={'color': TEXT_MUTED, 'margin': '4px 0 0 0', 'fontSize': '10px', 'letterSpacing': '0.5px'})], style=kpi_panel_style),
            html.Div([html.H2(movies_count, style={'color': SPOTIFY_GREEN, 'margin': '0', 'fontSize': '16px', 'fontWeight': 'bold'}), html.P("TOP PERFORMING AUDIO", style={'color': TEXT_MUTED, 'margin': '4px 0 0 0', 'fontSize': '10px', 'letterSpacing': '0.5px'})], style=kpi_panel_style),
            html.Div([html.H2(tv_seasons, style={'color': TEXT_MAIN, 'margin': '0', 'fontSize': '18px'}), html.P("CHAMPION CUMULATIVE VOL", style={'color': TEXT_MUTED, 'margin': '4px 0 0 0', 'fontSize': '10px', 'letterSpacing': '0.5px'})], style=kpi_panel_style),
            html.Div([html.H2(territories, style={'color': TEXT_MAIN, 'margin': '0', 'fontSize': '18px'}), html.P("ACTIVE CATALOG TITLES", style={'color': TEXT_MUTED, 'margin': '4px 0 0 0', 'fontSize': '10px', 'letterSpacing': '0.5px'})], style=kpi_panel_style),
        ], style={'display': 'flex', 'gap': '12px', 'width': '63%'}),

        # Power BI Styled Dropdown Slicer Simulation
        html.Div([
            html.Div([
                html.Button("Qtr 1", style=slicer_btn),
                html.Button("Qtr 2", style=slicer_btn),
                html.Button("Qtr 3", style=slicer_btn),
                html.Button("Qtr 4", style=slicer_btn),
                html.Button("All Time", style=slicer_active),
            ], style={'display': 'flex', 'gap': '4px'}),
            dcc.Dropdown(options=['Global All', 'Americas', 'EMEA', 'APAC'], value='Global All', clearable=False, style={'width': '105px', 'fontSize': '11px'})
        ], style={**panel_layout_style, 'width': '35%', 'marginBottom': '0px', 'display': 'flex', 'justifyContent': 'space-between', 'alignItems': 'center', 'padding': '10px 15px'})

    ], style={'display': 'flex', 'justifyContent': 'space-between', 'gap': '15px', 'marginBottom': '20px'}),

    # --- GRID VIEW PORT ROW 1 (3-COLUMN MATRIX LAYOUT) ---
    html.Div([
        html.Div([dcc.Graph(figure=fig_state, config={'displayModeBar': False}, style={'height': '290px'})], style={**panel_layout_style, 'width': '32%'}),
        html.Div([dcc.Graph(figure=fig_category, config={'displayModeBar': False}, style={'height': '290px'})], style={**panel_layout_style, 'width': '32%'}),
        html.Div([dcc.Graph(figure=fig_subcat, config={'displayModeBar': False}, style={'height': '290px'})], style={**panel_layout_style, 'width': '32%'})
    ], style={'display': 'flex', 'justifyContent': 'space-between', 'gap': '12px'}),

    # --- GRID VIEW PORT ROW 2 (2-COLUMN MATRIX LAYOUT) ---
    html.Div([
        html.Div([dcc.Graph(figure=fig_customer, config={'displayModeBar': False}, style={'height': '290px'})], style={**panel_layout_style, 'width': '49%'}),
        html.Div([dcc.Graph(figure=fig_payment, config={'displayModeBar': False}, style={'height': '290px'})], style={**panel_layout_style, 'width': '49%'})
    ], style={'display': 'flex', 'justifyContent': 'space-between', 'gap': '12px'})

], style={
    'backgroundColor': CANVAS_BG, 'padding': '20px 25px',
    'fontFamily': 'Helvetica Neue, Arial, sans-serif', 'minHeight': '100vh'
})

# =========================
# STEP 7 : BACKGROUND DAEMON RUNNER
# =========================
PORT = 8112

def run_dash():
    app.run(host='0.0.0.0', port=PORT, debug=False, use_reloader=False)

thread = threading.Thread(target=run_dash)
thread.daemon = True
thread.start()

# Mount the interactive iframe display straight into the Colab cell interface output window
output.serve_kernel_port_as_iframe(PORT, width='100%', height='800px')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 34.7 MB/s eta 0:00:00
⚠️ '/content/spotify.csv' not detected. Creating clean runtime baseline fallback...
Dash is running on http://0.0.0.0:8112/



INFO:dash.dash:Dash is running on http://0.0.0.0:8112/



<IPython.core.display.Javascript object>